# LabelInspect — first locked real-test evaluation

This notebook evaluates the trained DTU-Net/Tsimplex checkpoint on the held-out
real printed-label test set. The model, partial-diffusion distance, random seed,
pixel threshold and image threshold were fixed before this dataset was opened by
the model. Ground-truth masks were also created before any real-test prediction.

Attach exactly two Kaggle inputs:

1. `printed_label_test_locked_v1.zip`
2. `printed_label_latest.pt`

Select a GPU accelerator, enable Internet, and run all cells. Expected runtime on
a Tesla T4 is roughly 4–8 minutes after dependencies and the first Tsimplex
compilation complete. Download `printed_label_final_evaluation_results.zip` from
the Kaggle working files when the final cell finishes.

The notebook reports model-only results on 53 registered images. Two additional
severe-tear photographs that could not be registered are reported separately as
registration-stage rejects and are never described as DTU-Net detections.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, importlib.util, json, math, random, shutil, subprocess, sys, time, zipfile

import numpy as np
import torch
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/labelinspect')
WORK.mkdir(parents=True,exist_ok=True)
OUTPUT=WORK/'printed_label_final_evaluation'
OUTPUT.mkdir(parents=True,exist_ok=True)

def find_test_datasets(search_root):
    found=[]
    for manifest in search_root.rglob('manifest.csv'):
        candidate=manifest.parent
        if (candidate/'images'/'normal').is_dir() and (candidate/'ground_truth'/'tear').is_dir():
            found.append(candidate)
    return sorted(set(found))

dataset_candidates=find_test_datasets(KAGGLE_INPUT)
if not dataset_candidates:
    matching=[]
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names=['/'+item.filename.replace('\\','/').lstrip('/') for item in archive.infolist()]
                if any('/printed_label_test_locked_v1/manifest.csv' in name for name in names):
                    matching.append(archive_path)
        except zipfile.BadZipFile:
            pass
    if len(matching)==1:
        extraction_root=WORK/'uploaded_real_test';extraction_root.mkdir(parents=True,exist_ok=True)
        resolved=extraction_root.resolve()
        with zipfile.ZipFile(matching[0]) as archive:
            for item in archive.infolist():
                target=(extraction_root/item.filename).resolve()
                if target!=resolved and resolved not in target.parents:
                    raise ValueError(f'Unsafe ZIP member: {item.filename}')
            archive.extractall(extraction_root)
        dataset_candidates=find_test_datasets(extraction_root)
if len(dataset_candidates)!=1:
    raise FileNotFoundError('Expected one locked printed-label test dataset; found: '+repr([str(p) for p in dataset_candidates]))
DATA_ROOT=dataset_candidates[0]

checkpoint_candidates=sorted(set(KAGGLE_INPUT.rglob('printed_label_latest.pt')) | set(KAGGLE_INPUT.rglob('latest.pt')))
if not checkpoint_candidates:
    extracted_roots=[]
    for data_pickle in KAGGLE_INPUT.rglob('data.pkl'):
        candidate=data_pickle.parent
        if (candidate/'data').is_dir() and (candidate/'version').is_file():
            extracted_roots.append(candidate)
    extracted_roots=sorted(set(extracted_roots))
    if len(extracted_roots)==1:
        archive_root=extracted_roots[0]
        rebuilt=WORK/'rebuilt_printed_label_checkpoint.pt'
        with zipfile.ZipFile(rebuilt,'w',compression=zipfile.ZIP_STORED) as archive:
            for source_file in sorted(archive_root.rglob('*')):
                if source_file.is_file():
                    archive.write(source_file,f'{archive_root.name}/{source_file.relative_to(archive_root).as_posix()}')
        checkpoint_candidates=[rebuilt]
        print('Rebuilt Kaggle-extracted checkpoint:',rebuilt)
if len(checkpoint_candidates)!=1:
    raise FileNotFoundError('Expected one printed-label checkpoint; found: '+repr([str(p) for p in checkpoint_candidates]))
CHECKPOINT=checkpoint_candidates[0]

FROZEN_SELECTION=json.loads(r'''{"distance":50,"pixel_threshold":0.023506546393036842,"image_threshold":0.030804647132754326,"normal_pixel_fpr":0.004998696558915537,"normal_image_fpr":0.0,"synthetic_mean_dice":0.531818556586623,"synthetic_mean_iou":0.429141885251991,"synthetic_image_sensitivity":0.4666666666666667,"mean_seconds_per_image":3.6049217822499955,"by_kind":{"missing_print":{"mean_dice":0.5460377590335805,"image_sensitivity":0.2},"smudge":{"mean_dice":0.8430374586294755,"image_sensitivity":1.0},"tear":{"mean_dice":0.20638045209681283,"image_sensitivity":0.2}}}''')
SEED=230274
IMAGE_SIZE=224
BATCH_SIZE=4
T_DISTANCE=50
PIXEL_THRESHOLD=0.023506546393036842
IMAGE_SCORE_QUANTILE=0.995
IMAGE_THRESHOLD=0.030804647132754326
assert FROZEN_SELECTION['distance']==T_DISTANCE
assert abs(FROZEN_SELECTION['pixel_threshold']-PIXEL_THRESHOLD)<1e-12
assert abs(FROZEN_SELECTION['image_threshold']-IMAGE_THRESHOLD)<1e-12
print('GPU:',torch.cuda.get_device_name(0))
print('Locked test dataset:',DATA_ROOT)
print('Checkpoint:',CHECKPOINT)
print('Frozen distance and thresholds:',T_DISTANCE,PIXEL_THRESHOLD,IMAGE_THRESHOLD)


In [ ]:
missing=[]
for package,module in [('timm','timm'),('einops','einops'),('numba','numba'),('scikit-learn','sklearn')]:
    if importlib.util.find_spec(module) is None: missing.append(package)
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Dependencies ready.')


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
spec=importlib.util.spec_from_file_location('labelinspect_author_smoke',WORK/'author_smoke.py')
author=importlib.util.module_from_spec(spec);sys.modules[spec.name]=author;spec.loader.exec_module(author)
import numba
numba.set_num_threads(min(2,numba.get_num_threads()))
source_cache=WORK/'upstream'/author.COMMIT
hashes=author.fetch_sources(source_cache)
model_module,diffusion_ns,Adapter,compatibility=author.load_author_components(source_cache,OUTPUT)
print('Pinned author commit:',author.COMMIT)


In [ ]:
def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()

protocol=json.loads((DATA_ROOT/'protocol.json').read_text())
assert protocol['status']=='locked_for_first_real_test',protocol
with (DATA_ROOT/'manifest.csv').open(newline='',encoding='utf8') as stream:
    records=list(csv.DictReader(stream))
expected_counts={'normal':10,'missing_print':15,'smudge':15,'tear':13}
assert Counter(row['category'] for row in records)==Counter(expected_counts),Counter(row['category'] for row in records)
assert len(records)==53 and sum(int(row['is_anomaly']) for row in records)==43
assert len({row['image'] for row in records})==len(records)
for row in records:
    image_path=DATA_ROOT/row['image'];mask_path=DATA_ROOT/row['mask']
    assert image_path.is_file() and mask_path.is_file(),row
    assert sha256(image_path)==row['image_sha256'],image_path
    assert sha256(mask_path)==row['mask_sha256'],mask_path

RESAMPLE=getattr(Image,'Resampling',Image).BILINEAR
NEAREST=getattr(Image,'Resampling',Image).NEAREST
def pad_image(image,resample=RESAMPLE,fill=(238,238,238)):
    fitted=ImageOps.contain(image,(IMAGE_SIZE,IMAGE_SIZE),resample)
    canvas=Image.new(image.mode,(IMAGE_SIZE,IMAGE_SIZE),fill)
    canvas.paste(fitted,((IMAGE_SIZE-fitted.width)//2,(IMAGE_SIZE-fitted.height)//2))
    return canvas

def tensor_from_image(image):
    array=np.asarray(pad_image(image.convert('RGB')),dtype=np.float32).copy()/127.5-1.
    return torch.from_numpy(array).permute(2,0,1)

tensors=[];truth_masks=[]
for row in records:
    with Image.open(DATA_ROOT/row['image']) as opened:tensors.append(tensor_from_image(opened))
    with Image.open(DATA_ROOT/row['mask']) as opened:
        truth_masks.append(np.asarray(pad_image(opened.convert('L'),resample=NEAREST,fill=0))>0)
tensors=torch.stack(tensors)
truth_masks=np.stack(truth_masks)
image_labels=np.asarray([int(row['is_anomaly']) for row in records],dtype=bool)
categories=np.asarray([row['category'] for row in records])
names=np.asarray([Path(row['image']).stem for row in records])
physical_ids=np.asarray([row['physical_id'] for row in records])

label_roi=np.zeros((IMAGE_SIZE,IMAGE_SIZE),dtype=bool)
resized_height=round(650*IMAGE_SIZE/1063);roi_top=(IMAGE_SIZE-resized_height)//2
label_roi[roi_top:roi_top+resized_height,:]=True
assert tensors.shape==(53,3,224,224) and truth_masks.shape==(53,224,224)
assert not truth_masks[~image_labels].any()
assert all(mask.any() for mask in truth_masks[image_labels])
print('Hashes and masks verified for',len(records),'locked test images.')
print('Counts:',dict(Counter(categories)))


In [ ]:
device=torch.device('cuda:0')
checkpoint=torch.load(CHECKPOINT,map_location=device,weights_only=False)
assert checkpoint['step']==2000
assert checkpoint['author_commit']==author.COMMIT
assert checkpoint.get('dataset_category')=='printed_label_train_v1'
model_config=checkpoint['model_config']
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
backbone=model_module.UDHVT(**model_config).to(device);model=Adapter(backbone)
model.load_state_dict(checkpoint['model']);model.eval()
diffusion=diffusion_ns['GaussianDiffusionModel'](
    [224,224],diffusion_ns['get_beta_schedule'](1000,'cosine'),img_channels=3,
    loss_type='l2',noise='4dsimplex',octave=6,frequency=64,persistence=.9,train=False)
print('Compiling Tsimplex noise function on first use...')
diffusion.noise_fn(torch.zeros(1,1,4,4,device=device),torch.tensor([5],device=device))
torch.cuda.reset_peak_memory_stats()
print('Loaded frozen step',checkpoint['step'],'checkpoint from author commit',author.COMMIT)


In [ ]:
def reconstruct(inputs):
    outputs=[];batch_times=[]
    for start in range(0,len(inputs),BATCH_SIZE):
        x=inputs[start:start+BATCH_SIZE].to(device)
        tick=time.perf_counter()
        with torch.inference_mode():
            result=diffusion.forward_backward(model,x,None,see_whole_sequence=None,
                                               t_distance=T_DISTANCE,denoise_fn='noise_fn')
        torch.cuda.synchronize();batch_times.append(time.perf_counter()-tick)
        outputs.append(result.cpu())
        print(f'Reconstructed {min(start+BATCH_SIZE,len(inputs))}/{len(inputs)}')
    return torch.cat(outputs),batch_times

# This is the first model access to the locked real test images.
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
reconstructions,batch_times=reconstruct(tensors)
score_maps=(tensors-reconstructions).square().mean(dim=1).numpy().astype(np.float32)
predicted_masks=(score_maps>PIXEL_THRESHOLD)&label_roi[None]
image_scores=np.quantile(score_maps[:,label_roi],IMAGE_SCORE_QUANTILE,axis=1)
image_predictions=image_scores>IMAGE_THRESHOLD
mean_seconds_per_image=float(sum(batch_times)/len(records))
print('Inference complete; mean seconds/image:',mean_seconds_per_image)


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve

def safe_div(numerator,denominator):
    return float(numerator/denominator) if denominator else None

def segmentation_metrics(pred,truth):
    pred=np.asarray(pred,dtype=bool);truth=np.asarray(truth,dtype=bool)
    tp=int(np.sum(pred&truth));fp=int(np.sum(pred&~truth));fn=int(np.sum(~pred&truth));tn=int(np.sum(~pred&~truth))
    return {
        'tp':tp,'fp':fp,'fn':fn,'tn':tn,
        'dice':safe_div(2*tp,2*tp+fp+fn),
        'iou':safe_div(tp,tp+fp+fn),
        'precision':safe_div(tp,tp+fp),
        'recall':safe_div(tp,tp+fn),
    }

per_image=[]
for index,row in enumerate(records):
    segmentation=segmentation_metrics(predicted_masks[index]&label_roi,truth_masks[index]&label_roi)
    per_image.append({
        'file':names[index],
        'category':categories[index],
        'physical_id':physical_ids[index],
        'capture_condition':row['capture_condition'],
        'is_anomaly':int(image_labels[index]),
        'image_score':float(image_scores[index]),
        'image_detected':int(image_predictions[index]),
        **segmentation,
    })

tp_img=int(np.sum(image_predictions&image_labels));tn_img=int(np.sum(~image_predictions&~image_labels))
fp_img=int(np.sum(image_predictions&~image_labels));fn_img=int(np.sum(~image_predictions&image_labels))
image_summary={
    'tp':tp_img,'tn':tn_img,'fp':fp_img,'fn':fn_img,
    'auroc':float(roc_auc_score(image_labels,image_scores)),
    'average_precision':float(average_precision_score(image_labels,image_scores)),
    'sensitivity':safe_div(tp_img,tp_img+fn_img),
    'specificity':safe_div(tn_img,tn_img+fp_img),
    'precision':safe_div(tp_img,tp_img+fp_img),
    'f1':safe_div(2*tp_img,2*tp_img+fp_img+fn_img),
    'balanced_accuracy':float((safe_div(tp_img,tp_img+fn_img)+safe_div(tn_img,tn_img+fp_img))/2),
}

roi_truth=truth_masks[:,label_roi].reshape(-1)
roi_scores=score_maps[:,label_roi].reshape(-1)
pixel_summary={
    'auroc':float(roc_auc_score(roi_truth,roi_scores)),
    'average_precision':float(average_precision_score(roi_truth,roi_scores)),
    'normal_pixel_fpr':float(predicted_masks[~image_labels][:,label_roi].mean()),
}
defect_rows=[row for row in per_image if row['is_anomaly']]
pixel_summary['mean_dice_defect_images']=float(np.mean([row['dice'] for row in defect_rows]))
pixel_summary['mean_iou_defect_images']=float(np.mean([row['iou'] for row in defect_rows]))
pixel_summary['mean_pixel_recall_defect_images']=float(np.mean([row['recall'] for row in defect_rows]))

by_category={}
normal_indices=np.flatnonzero(categories=='normal')
for category in ['missing_print','smudge','tear']:
    indices=np.flatnonzero(categories==category)
    classification_indices=np.concatenate([normal_indices,indices])
    subset=[per_image[index] for index in indices]
    subset_truth=truth_masks[indices][:,label_roi].reshape(-1)
    subset_scores=score_maps[indices][:,label_roi].reshape(-1)
    by_category[category]={
        'image_count':len(indices),
        'physical_label_count':len(set(physical_ids[indices])),
        'image_sensitivity':float(np.mean(image_predictions[indices])),
        'image_auroc_vs_normals':float(roc_auc_score(image_labels[classification_indices],image_scores[classification_indices])),
        'image_average_precision_vs_normals':float(average_precision_score(image_labels[classification_indices],image_scores[classification_indices])),
        'mean_dice':float(np.mean([row['dice'] for row in subset])),
        'mean_iou':float(np.mean([row['iou'] for row in subset])),
        'pixel_auroc_within_defect_images':float(roc_auc_score(subset_truth,subset_scores)),
        'pixel_average_precision_within_defect_images':float(average_precision_score(subset_truth,subset_scores)),
    }

physical_rows=[]
for label_id in sorted(set(physical_ids)):
    indices=np.flatnonzero(physical_ids==label_id)
    detections=int(np.sum(image_predictions[indices]));total=len(indices)
    physical_rows.append({
        'physical_id':label_id,
        'category':categories[indices[0]],
        'capture_count':total,
        'detected_capture_count':detections,
        'capture_detection_rate':float(detections/total),
        'any_capture_detected':int(detections>0),
        'majority_captures_detected':int(detections>=math.ceil(total/2)),
        'median_image_score':float(np.median(image_scores[indices])),
        'max_image_score':float(np.max(image_scores[indices])),
    })

# These two inputs were rejected before the model because no defensible
# registration was possible. Keep hybrid accounting distinct from model metrics.
registration_rejected_anomalies=2
hybrid_tp=tp_img+registration_rejected_anomalies
hybrid_fn=fn_img
hybrid_summary={
    'registered_model_tp':tp_img,
    'registration_stage_tp':registration_rejected_anomalies,
    'total_anomaly_photographs':45,
    'sensitivity':safe_div(hybrid_tp,hybrid_tp+hybrid_fn),
    'specificity':safe_div(tn_img,tn_img+fp_img),
    'warning':'Hybrid pipeline result; registration rejects are not DTU-Net detections.',
}

report={
    'status':'completed_first_locked_real_test',
    'scope':'single_fixed_design_printed_label_pilot',
    'paper_result_reproduced':False,
    'checkpoint_step':int(checkpoint['step']),
    'author_commit':author.COMMIT,
    'registered_test_images':len(records),
    'registered_normal_images':int(np.sum(~image_labels)),
    'registered_anomaly_images':int(np.sum(image_labels)),
    'physical_test_labels':len(set(physical_ids)),
    'frozen_configuration':{
        'seed':SEED,'t_distance':T_DISTANCE,'pixel_threshold':PIXEL_THRESHOLD,
        'image_score_quantile':IMAGE_SCORE_QUANTILE,'image_threshold':IMAGE_THRESHOLD,
    },
    'image_level_model_only':image_summary,
    'pixel_level_model_only':pixel_summary,
    'by_category':by_category,
    'end_to_end_hybrid':hybrid_summary,
    'mean_inference_seconds_per_image':mean_seconds_per_image,
    'peak_gpu_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
    'peak_gpu_reserved_gib':torch.cuda.max_memory_reserved()/2**30,
    'gpu':torch.cuda.get_device_name(0),
    'limitations':[
        'The 43 registered anomaly images are repeated captures of nine physical defects.',
        'Missing-print samples are white-paper covered-print simulations.',
        'Smudge masks conservatively include clearly added visible strokes away from expected print.',
        'Two severe tear photographs were rejected before model inference because registration was underdetermined.',
        'This pilot does not reproduce the paper training scale or reported benchmark result.',
    ],
}

(OUTPUT/'evaluation_report.json').write_text(json.dumps(report,indent=2))
with (OUTPUT/'per_image.csv').open('w',newline='') as stream:
    writer=csv.DictWriter(stream,fieldnames=list(per_image[0]));writer.writeheader();writer.writerows(per_image)
with (OUTPUT/'per_physical_label.csv').open('w',newline='') as stream:
    writer=csv.DictWriter(stream,fieldnames=list(physical_rows[0]));writer.writeheader();writer.writerows(physical_rows)
np.savez_compressed(OUTPUT/'test_maps_float16.npz',names=names,scores=score_maps.astype(np.float16),
                    predictions=predicted_masks.astype(np.uint8),ground_truth=truth_masks.astype(np.uint8))
print(json.dumps(report,indent=2))


In [ ]:
def show_tensor(tensor):
    return ((tensor.permute(1,2,0).numpy()+1)/2).clip(0,1)

selected_indices=[
    int(np.flatnonzero(categories=='normal')[0]),
    int(np.flatnonzero(physical_ids=='M01')[0]),
    int(np.flatnonzero(physical_ids=='S01')[0]),
    int(np.flatnonzero(physical_ids=='T01')[0]),
]
fig,axes=plt.subplots(4,5,figsize=(15,10))
for row,index in enumerate(selected_indices):
    axes[row,0].imshow(show_tensor(tensors[index]));axes[row,0].set_title(f'{names[index]} | input')
    axes[row,1].imshow(show_tensor(reconstructions[index]));axes[row,1].set_title('Reconstruction')
    axes[row,2].imshow(score_maps[index],cmap='magma');axes[row,2].set_title(f'Residual | {image_scores[index]:.4f}')
    axes[row,3].imshow(truth_masks[index],cmap='gray',vmin=0,vmax=1);axes[row,3].set_title('Locked ground truth')
    axes[row,4].imshow(predicted_masks[index],cmap='gray',vmin=0,vmax=1);axes[row,4].set_title(f'Prediction | detected={bool(image_predictions[index])}')
    for axis in axes[row]:axis.axis('off')
fig.suptitle('First locked real printed-label evaluation | frozen t=50')
fig.tight_layout();fig.savefig(OUTPUT/'evaluation_preview.png',dpi=160,bbox_inches='tight');plt.show()

fpr,tpr,_=roc_curve(image_labels,image_scores)
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
axes[0].plot(fpr,tpr,label=f"AUROC = {image_summary['auroc']:.3f}")
axes[0].plot([0,1],[0,1],'--',color='gray');axes[0].set(xlabel='False-positive rate',ylabel='True-positive rate',title='Image-level ROC')
axes[0].legend();axes[0].grid(alpha=.25)
score_groups=[image_scores[categories==category] for category in ['normal','missing_print','smudge','tear']]
axes[1].boxplot(score_groups,tick_labels=['normal','missing','smudge','tear'],showfliers=True)
axes[1].axhline(IMAGE_THRESHOLD,color='red',linestyle='--',label='frozen threshold')
axes[1].set(ylabel='99.5th-percentile residual',title='Frozen image scores by category')
axes[1].legend();axes[1].grid(alpha=.25,axis='y')
fig.tight_layout();fig.savefig(OUTPUT/'image_score_diagnostics.png',dpi=160,bbox_inches='tight');plt.show()

archive=shutil.make_archive('/kaggle/working/printed_label_final_evaluation_results','zip',OUTPUT)
print('\nLOCKED REAL TEST EVALUATION COMPLETED')
print('Download from Kaggle working files:',archive)
